# Colab smoke test — ovarian cancer prognosis

Proves the whole Colab path works end to end: GPU attached, this repo present on
the runtime, a tracked dataset loaded, and a model trained on the GPU.

**Before running**

1. Locally: `.venv/bin/python agent/scripts/colab_check.py --dataset gse14764-ovarian-expression-series-matrix`
   and fix anything it reports. The runtime clones the **remote**, so uncommitted
   or unpushed work is invisible to it.
2. In Cursor: *Select Kernel* → *Colab* → *New Colab Server* → **GPU**.
3. If the GitHub repo is private, add a personal access token as a Colab secret
   named `GITHUB_TOKEN` (key icon in Colab's left sidebar, "Notebook access" on).

Runtime settings live in `.research/config.yaml` under `colab:`. Helper functions
live in `agent/scripts/colab_env.py`.

## 1. Bootstrap

Puts this repo on the runtime's disk and imports the helper module. The clone is
shallow, blobless and sparse, so it pulls a few MB of code rather than the 1.4 GB
`datasets/` tree — individual datasets are fetched on demand later.

Also works against a local kernel, where it finds the checkout instead of cloning.

In [ ]:
# Mirrors .research/config.yaml -> colab
REPO, BRANCH, DEST = "github.com/XooTB/research.git", "main", "/content/research"
SPARSE = ["agent", ".research", "docs", "notebooks"]
TOKEN_SECRET = "GITHUB_TOKEN"

import pathlib
import subprocess
import sys


def on_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


if on_colab():
    token = ""
    try:
        from google.colab import userdata
        token = userdata.get(TOKEN_SECRET) or ""
    except Exception:
        pass  # public repo, or the secret isn't shared with this notebook

    url = f"https://{token + '@' if token else ''}{REPO}"
    if not pathlib.Path(DEST, ".git").exists():
        try:
            subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                            "--sparse", "--branch", BRANCH, url, DEST],
                           check=True, capture_output=True, text=True)
        except subprocess.CalledProcessError as exc:
            detail = exc.stderr.replace(token, "***") if token else exc.stderr
            raise SystemExit(
                "clone failed. If the repo is private, add a PAT as the Colab "
                f"secret {TOKEN_SECRET!r} and enable notebook access.\n{detail}"
            ) from None
        subprocess.run(["git", "-C", DEST, "sparse-checkout", "set", *SPARSE], check=True)
    else:
        subprocess.run(["git", "-C", DEST, "pull", "--ff-only"], check=False)
    root = pathlib.Path(DEST)
else:
    root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
                if (p / "agent" / "scripts" / "colab_env.py").exists())

sys.path.insert(0, str(root / "agent" / "scripts"))
import colab_env as ce

print(ce.summary())

## 2. Dependencies

Colab already ships torch, pandas and scikit-learn, so `require` is usually a
no-op. `install_requirements()` adds the research-specific extras from
`agent/requirements-colab.txt` (lifelines, pymupdf).

In [ ]:
print(ce.require("pandas", "numpy", "scikit-learn", "torch"))
print(ce.install_requirements())

## 3. Load a tracked dataset

`geo_xy` widens the sparse-checkout to pull just this dataset, reads the CSVs
written by `agent/scripts/datasets_to_csv.py`, transposes the expression matrix
to samples x probes, and aligns the label by GSM accession.

GSE14764: 80 primary ovarian tumours, 22,283 Affymetrix probes, binary overall
survival event.

In [ ]:
SLUG = "gse14764-ovarian-expression-series-matrix"

X, y, meta = ce.geo_xy(SLUG, label="overall survival event")
print(meta)
X.iloc[:5, :5]

## 4. CPU baseline

Regularized logistic regression on the 500 most informative probes. Selection and
scaling sit inside the pipeline so they are refit per fold — otherwise the AUC
would be inflated by leakage from the held-out samples.

With 80 samples this is a sanity check on the plumbing, not a publishable result.

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

pipe = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    SelectKBest(f_classif, k=500),
    LogisticRegression(max_iter=5000, C=0.1),
)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
auc = cross_val_score(pipe, X.values, y.values, cv=cv, scoring="roc_auc")
print(f"logreg 5-fold CV ROC-AUC: {auc.mean():.3f} +/- {auc.std():.3f}")

## 5. Confirm the GPU is real

A matmul benchmark first, because "CUDA available" can be true while the
accelerator is unused. Expect a T4 to beat these CPU cores by 20-50x on a
4096x4096 matmul.

In [ ]:
import time

import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch {torch.__version__} -> {device}")
if device == "cpu":
    print("No GPU. Reconnect with a GPU runtime before reading anything into the timings.")


def bench(dev, n=4096, reps=3):
    a, b = torch.randn(n, n, device=dev), torch.randn(n, n, device=dev)
    if dev == "cuda":
        a @ b  # warm up: first CUDA call pays context setup
        torch.cuda.synchronize()
    start = time.perf_counter()
    for _ in range(reps):
        a @ b
    if dev == "cuda":
        torch.cuda.synchronize()
    return (time.perf_counter() - start) / reps


cpu_s = bench("cpu")
print(f"cpu  {cpu_s * 1e3:8.1f} ms")
if device == "cuda":
    gpu_s = bench("cuda")
    print(f"cuda {gpu_s * 1e3:8.1f} ms  ({cpu_s / gpu_s:.0f}x faster)")

## 6. Train on the GPU

A small MLP on the same features. 60 training samples against 500 features will
overfit — the point is that the CUDA training loop runs, not the held-out score.

In [ ]:
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from torch import nn

X_tr, X_te, y_tr, y_te = train_test_split(
    X.values, y.values, test_size=0.25, stratify=y.values, random_state=0)

prep = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                     SelectKBest(f_classif, k=500))
X_tr_t = prep.fit_transform(X_tr, y_tr)
X_te_t = prep.transform(X_te)

torch.manual_seed(0)


def to_device(a):
    return torch.tensor(a, dtype=torch.float32, device=device)


X_tr_g, X_te_g = to_device(X_tr_t), to_device(X_te_t)
y_tr_g = to_device(y_tr).unsqueeze(1)

model = nn.Sequential(
    nn.Linear(X_tr_t.shape[1], 64), nn.ReLU(), nn.Dropout(0.3),
    nn.Linear(64, 1),
).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)
loss_fn = nn.BCEWithLogitsLoss()

model.train()
for epoch in range(1, 201):
    opt.zero_grad()
    loss = loss_fn(model(X_tr_g), y_tr_g)
    loss.backward()
    opt.step()
    if epoch % 50 == 0:
        print(f"epoch {epoch:3d}  loss {loss.item():.4f}")

model.eval()
with torch.no_grad():
    probs = torch.sigmoid(model(X_te_g)).cpu().numpy().ravel()
mlp_auc = roc_auc_score(y_te, probs)
print(f"\nMLP held-out ROC-AUC ({len(y_te)} samples): {mlp_auc:.3f}")

## 7. Save the run

The runtime's disk is wiped when the session ends. `save_run` writes a JSON
record (result + full environment snapshot) into `.research/colab/runs/` inside
the runtime checkout; the next cell shows how to get it back to your machine.

In [ ]:
run_dir = ce.save_run("gse14764-os-baseline", {
    "dataset": meta,
    "logreg_cv_auc": {"mean": float(auc.mean()), "std": float(auc.std()),
                      "folds": [float(a) for a in auc]},
    "mlp_holdout_auc": float(mlp_auc),
    "device": device,
})
print(run_dir)
print((run_dir / "run.json").read_text()[:1200])

In [ ]:
import subprocess

# Pushing the record is what lets the agent read these results back
# (agent/scripts/colab_runs.py) instead of asking what the output said.
if not ce.in_colab():
    print("local kernel — nothing to push")
else:
    ws = str(ce.workspace())
    ident = ["-c", "user.name=colab", "-c", "user.email=colab@local"]
    for label, cmd in [
        ("add", ["git", "-C", ws, "add", ".research/colab/runs"]),
        ("commit", ["git", "-C", ws, *ident, "commit", "-m", f"colab: {run_dir.name}"]),
        ("push", ["git", "-C", ws, "push", "origin", "HEAD:main"]),
    ]:
        r = subprocess.run(cmd, capture_output=True, text=True)
        print(f"[{label}] {(r.stdout or r.stderr).strip() or 'ok'}")

## 8. Getting results back

The cell above already pushed the run record, which is what closes the loop: the
agent can now `git pull` and read these results with

```bash
.venv/bin/python agent/scripts/colab_runs.py --last
.venv/bin/python agent/scripts/colab_runs.py --compare   # across runs
```

Nothing else here survives the session, so for larger artifacts pick one:

**Copy to Google Drive** — better for model weights and anything large:

```python
import shutil
dest = ce.mount_drive()          # /content/drive/MyDrive/research-colab
dest.mkdir(parents=True, exist_ok=True)
shutil.copytree(run_dir, dest / run_dir.name, dirs_exist_ok=True)
```

**Download a single file** — quickest for one artifact:

```python
from google.colab import files
files.download(str(run_dir / "run.json"))
```

## Notes

- Cells run on Google's VM; the file lives on your disk. The Cursor agent can
  edit this notebook but cannot open a terminal on the runtime — use `!cmd` cells
  or `subprocess` for anything shell-shaped.
- Free-tier sessions get reclaimed (roughly 12h ceiling, much less when idle) and
  the GPU is usually a T4. Checkpoint long training runs to Drive.
- `datasets/` on the runtime only contains what `ensure_dataset` /
  `geo_xy` pulled in. Add more with `ce.ensure_dataset("<slug>")`.
- Files over 100 MB reach the runtime as `.zip` (or `.zip.partNN` when the zip
  is also oversize), never as the raw CSV. `ensure_dataset` unpacks them; call
  `ce.restore_packed()` if you read a dataset path directly.